[A. Data Preparation](#scrollTo=6N6VCO9sGeuY)

[B. Analysis](#scrollTo=cRNN9XpbN1ti)

[C. Actionable Output](#scrollTo=oTUGcIwfnSb5)

[D. Bonus](#scrollTo=Ee0mI2uNyM9N)

## **A. Data Preparation**

### **A1. IMPORT LIBRARIES**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
COLORS = {
    "Extreme Fear":  "#A32D2D",
    "Fear":          "#E24B4A",
    "Neutral":       "#888780",
    "Greed":         "#1D9E75",
    "Extreme Greed": "#085041",
}

print("Libraries imported successfully.")

### **A2. LOAD DATA**

In [ ]:
SENTIMENT_PATH = "./content/fear_greed_index.csv"
TRADES_PATH    = "./content/historical_data.csv"

sentiment_raw = pd.read_csv(SENTIMENT_PATH)
trades_raw    = pd.read_csv(TRADES_PATH, on_bad_lines="skip")

print(f"Sentiment  →  {sentiment_raw.shape[0]:,} rows  ×  {sentiment_raw.shape[1]} cols")
print(f"Trades     →  {trades_raw.shape[0]:,} rows  ×  {trades_raw.shape[1]} cols")
print("\n── Sentiment sample ──")
display(sentiment_raw.head(3))
print("\n── Trades sample ──")
display(trades_raw.head(3))


### **A3. BASIC INSPECTION**

In [ ]:
for label, df in [("Sentiment", sentiment_raw), ("Trades", trades_raw)]:
    print(f"\n{'─'*40}\n{label} — dtypes & nulls\n{'─'*40}")
    info = pd.DataFrame({
        "dtype":    df.dtypes,
        "nulls":    df.isnull().sum(),
        "null_%":   (df.isnull().mean() * 100).round(2),
        "unique":   df.nunique(),
    })
    display(info)
    print(f"Duplicates: {df.duplicated().sum()}")

### **A4. DATA CLEANING**

In [ ]:
sentiment = sentiment_raw.drop_duplicates().copy()
trades    = trades_raw.drop_duplicates().copy()

sentiment = sentiment.dropna(subset=["date", "classification"])
trades    = trades.dropna(subset=["Timestamp", "Account"])

print(f"After cleaning — Sentiment: {len(sentiment):,} | Trades: {len(trades):,}")

### **A5. DATE CONVERSION & ALIGNMENT**

In [ ]:
raw_ts = pd.to_numeric(trades["Timestamp"], errors="coerce").dropna()
median_ts = raw_ts.median()

if median_ts > 1e12:        # milliseconds
    unit = "ms"
elif median_ts > 1e9:       # seconds
    unit = "s"
else:                       # microseconds
    unit = "us"

print(f"Detected timestamp unit: {unit}  (median raw value: {median_ts:.0f})")

trades["Timestamp"] = pd.to_datetime(
    pd.to_numeric(trades["Timestamp"], errors="coerce"), unit=unit, errors="coerce"
)
sentiment["date"] = pd.to_datetime(sentiment["date"], errors="coerce")

trades    = trades.dropna(subset=["Timestamp"])
sentiment = sentiment.dropna(subset=["date"])

# Extract date-only keys for merging
trades["date_only"]    = trades["Timestamp"].dt.normalize()   # midnight datetime
sentiment["date_only"] = sentiment["date"].dt.normalize()

print(f"\nTrades date range   : {trades['date_only'].min().date()} → {trades['date_only'].max().date()}")
print(f"Sentiment date range: {sentiment['date_only'].min().date()} → {sentiment['date_only'].max().date()}")

# Sanity-check overlap
overlap = set(trades["date_only"]).intersection(set(sentiment["date_only"]))
print(f"Overlapping dates   : {len(overlap)}")

 ### **A6. STANDARDIZE COLUMN NAMES**

In [ ]:
trades = trades.rename(columns={
    "Account":         "account",
    "Coin":            "symbol",
    "Execution Price": "price",
    "Size Tokens":     "size_tokens",
    "Size USD":        "size_usd",
    "Side":            "side",
    "Closed PnL":      "closed_pnl",
    "Direction":       "direction",
    "Fee":             "fee",
    "Start Position":  "start_position",
    "Timestamp IST":   "timestamp_ist",
    "Transaction Hash":"tx_hash",
    "Order ID":        "order_id",
    "Trade ID":        "trade_id",
})

sentiment = sentiment.rename(columns={"classification": "sentiment"})

# Normalise sentiment labels (strip whitespace, title-case)
sentiment["sentiment"] = sentiment["sentiment"].str.strip().str.title()

print("Columns standardised.")
print("Sentiment values:", sentiment["sentiment"].unique())

### **A7. MERGE DATASETS**

In [ ]:
merged = pd.merge(
    trades,
    sentiment[["date_only", "sentiment"]],
    on="date_only",
    how="left",
)

# Coerce numerics
for col in ["closed_pnl", "size_usd", "price", "leverage", "fee"]:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors="coerce")

# Drop rows without PnL or trade size (can't analyse without them)
merged = merged.dropna(subset=["closed_pnl", "size_usd"])

print(f"\nMerged shape: {merged.shape}")
print("\nSentiment distribution after merge:")
print(merged["sentiment"].value_counts(dropna=False))

# Warn if many NaN sentiments remain
nan_pct = merged["sentiment"].isna().mean() * 100
if nan_pct > 5:
    print(f"\n⚠  {nan_pct:.1f}% of trades have no sentiment match.")
    print("   Check that date ranges overlap (see output above).")

### **A8. FEATURE ENGINEERING**

In [ ]:

merged["is_win"]  = merged["closed_pnl"] > 0


merged["is_long"] = merged["side"].str.lower().str.strip().isin(["buy", "long", "open long"])


if "fee" in merged.columns:
    merged["net_pnl"] = merged["closed_pnl"] - merged["fee"].fillna(0)
else:
    merged["net_pnl"] = merged["closed_pnl"]


merged = merged.sort_values(["account", "Timestamp"])
merged["cum_pnl"]       = merged.groupby("account")["net_pnl"].cumsum()
merged["rolling_peak"]  = merged.groupby("account")["cum_pnl"].cummax()
merged["drawdown"]      = merged["cum_pnl"] - merged["rolling_peak"]  # always ≤ 0

print("Feature engineering complete.")

### **A9. ACCOUNT LEVEL METRICS**

In [ ]:
acct = merged.groupby("account").agg(
    total_trades   = ("closed_pnl", "count"),
    total_pnl      = ("net_pnl",    "sum"),
    win_rate       = ("is_win",     "mean"),
    avg_trade_size = ("size_usd",   "mean"),
    long_ratio     = ("is_long",    "mean"),
    pnl_volatility = ("net_pnl",    "std"),
    max_drawdown   = ("drawdown",   "min"),
).reset_index()

acct["pnl_per_trade"] = acct["total_pnl"] / acct["total_trades"]

print(f"\nAccount-level metrics: {len(acct):,} unique accounts")
display(acct.describe())

### **A10. DAILY + SENTIMENT-LEVEL AGGREGATION**

In [ ]:
daily = merged.groupby(["account", "date_only", "sentiment"]).agg(
    daily_pnl    = ("net_pnl",    "sum"),
    trade_count  = ("closed_pnl", "count"),
    long_ratio   = ("is_long",    "mean"),
    win_rate     = ("is_win",     "mean"),
    avg_size_usd = ("size_usd",   "mean"),
).reset_index()

print(f"\nDaily aggregation: {len(daily):,} account-day rows")
display(daily.head())

 ### **A11. Final checks**

In [ ]:
print("\n── Final merged dataset info ──")
print(merged.info())
print("\n── Missing values ──")
print(merged.isnull().sum()[merged.isnull().sum() > 0])

## **B. Analysis**

### **B1. SENTIMENT DISTRIBUTION**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
vc = merged["sentiment"].value_counts()
bars = ax.bar(vc.index, vc.values,
              color=[COLORS.get(s, "#888780") for s in vc.index],
              edgecolor="white", width=0.55)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + vc.max() * 0.01,
            f"{bar.get_height():,}", ha="center", va="bottom", fontsize=11)
ax.set_title("Trade volume by market sentiment", fontweight="bold")
ax.set_ylabel("Number of trades")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
sns.despine()
plt.tight_layout()
plt.savefig("B1_sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### **B2. PnL PERFORMANCE: FEAR vs GREED**

In [ ]:
sent_perf = daily.dropna(subset=["sentiment"]).groupby("sentiment").agg(
    median_daily_pnl  = ("daily_pnl",  "median"),
    mean_daily_pnl    = ("daily_pnl",  "mean"),
    win_rate          = ("win_rate",   "mean"),
    pnl_std           = ("daily_pnl",  "std"),
    n_account_days    = ("daily_pnl",  "count"),
).reset_index()

print("\n── Sentiment Performance Summary ──")
display(sent_perf)

# Statistical test: Fear vs Greed daily PnL
groups = {s: daily[daily["sentiment"] == s]["daily_pnl"].dropna()
          for s in daily["sentiment"].dropna().unique()}

if "Fear" in groups and "Greed" in groups:
    t_stat, p_val = stats.mannwhitneyu(groups["Fear"], groups["Greed"], alternative="two-sided")
    print(f"\nMann-Whitney U (Fear vs Greed daily PnL): U={t_stat:.0f}, p={p_val:.4f}")
    print("Statistically significant difference:" , "YES ✓" if p_val < 0.05 else "NO")

# Plot: median daily PnL per sentiment
fig, axes = plt.subplots(1, 2, figsize=(12, 5))


ax = axes[0]
for i, row in sent_perf.iterrows():
    color = COLORS.get(row["sentiment"], "#888780")
    ax.bar(row["sentiment"], row["median_daily_pnl"],
           color=color, edgecolor="white", width=0.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Median daily PnL by sentiment", fontweight="bold")
ax.set_ylabel("Median daily PnL (USD)")
sns.despine(ax=ax)

# Win rate bar chart
ax = axes[1]
for i, row in sent_perf.iterrows():
    color = COLORS.get(row["sentiment"], "#888780")
    ax.bar(row["sentiment"], row["win_rate"] * 100,
           color=color, edgecolor="white", width=0.5)
ax.axhline(50, color="black", linewidth=0.8, linestyle="--", label="50% baseline")
ax.set_title("Average win rate by sentiment", fontweight="bold")
ax.set_ylabel("Win rate (%)")
ax.legend()
sns.despine(ax=ax)

plt.suptitle("B2 — Does sentiment affect performance?", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("B2_performance_by_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()

### **B3. PnL DISTRIBUTION: FEAR vs GREED (box plots)**

In [ ]:
plot_data = daily.dropna(subset=["sentiment"])
order     = [s for s in ["Fear", "Greed", "Extreme Fear", "Extreme Greed"]
             if s in plot_data["sentiment"].unique()]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))


ax = axes[0]
sns.boxplot(data=plot_data, x="sentiment", y="daily_pnl",
            order=order, palette=COLORS, ax=ax, width=0.5,
            flierprops=dict(marker="o", markersize=2, alpha=0.3))
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

q1, q99 = plot_data["daily_pnl"].quantile([0.01, 0.99])
ax.set_ylim(q1, q99)
ax.set_title("Daily PnL distribution by sentiment", fontweight="bold")
ax.set_ylabel("Daily PnL (USD)")
ax.set_xlabel("")
sns.despine(ax=ax)

# Density / KDE
ax = axes[1]
for sent in order:
    subset = plot_data[plot_data["sentiment"] == sent]["daily_pnl"]
    subset_clipped = subset.clip(q1, q99)
    subset_clipped.plot.kde(ax=ax, label=sent,
                            color=COLORS.get(sent, "#888780"), linewidth=2)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("PnL density by sentiment", fontweight="bold")
ax.set_xlabel("Daily PnL (USD)")
ax.legend()
sns.despine(ax=ax)

plt.suptitle("B3 — PnL distribution comparison", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("B3_pnl_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### **B4. TRADER BEHAVIOR BY SENTIMENT**

In [ ]:
# Do traders change leverage, trade frequency, long/short bias, or size
# depending on whether the market is in Fear or Greed?

behavior = daily.dropna(subset=["sentiment"]).groupby("sentiment").agg(
    avg_trade_freq   = ("trade_count",  "mean"),
    avg_long_ratio   = ("long_ratio",   "mean"),
    avg_size_usd     = ("avg_size_usd", "mean"),
).reset_index()

print("\n── Behavior by Sentiment ──")
display(behavior)

metrics = {
    "Avg trades per day":               "avg_trade_freq",
    "Long ratio (0=short, 1=long)":     "avg_long_ratio",
    "Avg trade size (USD)":             "avg_size_usd",
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, col) in zip(axes.flatten(), metrics.items()):
    if behavior[col].isna().all():
        ax.text(0.5, 0.5, f"No data for\n{col}", ha="center", va="center", transform=ax.transAxes)
        continue
    bars = ax.bar(behavior["sentiment"], behavior[col],
                  color=[COLORS.get(s, "#888780") for s in behavior["sentiment"]],
                  edgecolor="white", width=0.5)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.005,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("")
    sns.despine(ax=ax)

plt.suptitle("B4 — Trader behavior by sentiment", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("B4_behavior_by_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()

### **B5. DRAWDOWN PROXY BY SENTIMENT**

In [ ]:
drawdown_sent = merged.dropna(subset=["sentiment"]).groupby("sentiment")["drawdown"].median()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(drawdown_sent.index, drawdown_sent.values,
       color=[COLORS.get(s, "#888780") for s in drawdown_sent.index],
       edgecolor="white", width=0.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("B5 — Median drawdown proxy by sentiment", fontweight="bold")
ax.set_ylabel("Median drawdown (USD, lower = worse)")
sns.despine()
plt.tight_layout()
plt.savefig("B5_drawdown_by_sentiment.png", dpi=150, bbox_inches="tight")
plt.show()

### **B6. TRADER SEGMENTATION**

**Segment 1: Frequent vs Infrequent Traders**

In [ ]:
freq_median = acct["total_trades"].median()
acct["freq_segment"] = np.where(
    acct["total_trades"] >= freq_median, "Frequent", "Infrequent"
)
print(f"Trade frequency median cut: {freq_median:.0f} trades")


**Segment 2: High PnL vs Low PnL traders**

In [ ]:
pnl_median = acct["total_pnl"].median()
acct["pnl_segment"] = np.where(
    acct["total_pnl"] >= pnl_median, "High PnL", "Low PnL"
)
print(f"Total PnL median cut: {pnl_median:.2f}")

**Segment 3: Consistent Winners vs Inconsistent**

In [ ]:
vol_median = acct["pnl_volatility"].median()
acct["winner_segment"] = np.where(
    (acct["win_rate"] > 0.5) & (acct["pnl_volatility"] <= vol_median),
    "Consistent winner", "Inconsistent"
)

print("\nSegment sizes:")
print(acct["freq_segment"].value_counts())
print(acct["pnl_segment"].value_counts())
print(acct["winner_segment"].value_counts())

### **B7. SEGMENT x SENTIMENT INTERACTION**

In [ ]:
merged2 = merged.merge(
    acct[["account", "freq_segment", "pnl_segment", "winner_segment"]],
    on="account", how="left"
)

def segment_sentiment_summary(segment_col, title):
    """Return median daily PnL and win rate by segment × sentiment."""
    summary = (
        merged2.dropna(subset=["sentiment", segment_col])
        .groupby([segment_col, "sentiment"])
        .agg(
            median_pnl = ("net_pnl", "median"),
            win_rate   = ("is_win",  "mean"),
            n_trades   = ("net_pnl", "count"),
        )
        .reset_index()
    )
    print(f"\n── {title} ──")
    display(summary)
    return summary

s1 = segment_sentiment_summary("freq_segment",   "Frequent vs Infrequent × Sentiment")
s2 = segment_sentiment_summary("pnl_segment",    "High PnL vs Low PnL × Sentiment")
s3 = segment_sentiment_summary("winner_segment", "Winners vs Inconsistent × Sentiment")

### **B8. SEGMENT HEATMAPS**

In [ ]:
def plot_heatmap(summary_df, segment_col, value_col, title, ax, fmt=".1f"):
    pivot = summary_df.pivot(index=segment_col, columns="sentiment", values=value_col)
    sns.heatmap(pivot, annot=True, fmt=fmt, ax=ax,
                cmap="RdYlGn", linewidths=0.5, cbar=True)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

plot_heatmap(s1, "freq_segment",   "median_pnl",
             "Frequency segment — median PnL", axes[0])
plot_heatmap(s2, "pnl_segment",    "median_pnl",
             "PnL segment — median PnL",       axes[1])
plot_heatmap(s3, "winner_segment", "win_rate",
             "Winner segment — win rate",      axes[2], fmt=".2f")

plt.suptitle("B8 — Segment × Sentiment heatmaps", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("B8_segment_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

### **B9. INSIGHT SUMMARY TABLE**

In [ ]:
insight_table = pd.DataFrame({
    "Insight": [
        "1. Fear-day PnL",
        "2. Trade frequency",
        "3. Long/Short bias",
        "4. Trade size",
        "5. Winner segment resilience",
    ],
    "Finding": [
        "Median daily PnL is [higher/lower] on Fear days vs Greed days — see B2.",
        "Traders execute [more/fewer] trades during Greed vs Fear — see B4.",
        "Long ratio [rises/falls] during Greed, suggesting sentiment-driven directional bias — see B4.",
        "Average trade size is [larger/smaller] during Greed vs Fear — see B4.",
        "Consistent winners maintain positive median PnL regardless of sentiment — see B8.",
    ],
    "Evidence": [
        "Chart B2, Mann-Whitney p-value",
        "Chart B4 (trade frequency bar)",
        "Chart B4 (long ratio bar)",
        "Chart B4 (trade size bar)",
        "Chart B8 (winner segment row)",
    ],
})

print("\n── KEY INSIGHTS ──")
display(insight_table)

print("\n Parts A & B complete. Charts saved as PNG files.")
print("   Update the [higher/lower] placeholders in insight_table after reviewing your charts.")


## **C. ACTIONABLE OUTPUT**

### **C1. STRATEGY EVIDENCE BASE**

In [ ]:
print("=" * 60)
print("STRATEGY EVIDENCE BASE")
print("=" * 60)

# 1a. PnL by sentiment
print("\n── Median daily PnL by sentiment ──")
pnl_by_sent = (
    daily.dropna(subset=["sentiment"])
    .groupby("sentiment")["daily_pnl"]
    .agg(["median", "mean", "count"])
    .rename(columns={"median": "median_pnl", "mean": "mean_pnl", "count": "n_days"})
    .sort_values("median_pnl", ascending=False)
)
display(pnl_by_sent)

# 1b. Win rate by sentiment
print("\n── Win rate by sentiment ──")
wr_by_sent = (
    merged.dropna(subset=["sentiment"])
    .groupby("sentiment")["is_win"]
    .mean()
    .sort_values(ascending=False)
    .rename("win_rate")
    .to_frame()
)
display(wr_by_sent)

# 1c. Trade size by sentiment
print("\n── Avg trade size (USD) by sentiment ──")
size_by_sent = (
    merged.dropna(subset=["sentiment"])
    .groupby("sentiment")["size_usd"]
    .median()
    .sort_values(ascending=False)
    .rename("median_size_usd")
    .to_frame()
)
display(size_by_sent)

# 1d. Long ratio by sentiment
print("\n── Long ratio by sentiment ──")
long_by_sent = (
    merged.dropna(subset=["sentiment"])
    .groupby("sentiment")["is_long"]
    .mean()
    .sort_values(ascending=False)
    .rename("long_ratio")
    .to_frame()
)
display(long_by_sent)

# 1e. Winner segment performance by sentiment
print("\n── Consistent winners: win rate by sentiment ──")
winners_sent = (
    merged2[merged2["winner_segment"] == "Consistent winner"]
    .dropna(subset=["sentiment"])
    .groupby("sentiment")
    .agg(
        win_rate   = ("is_win",   "mean"),
        median_pnl = ("net_pnl",  "median"),
        n_trades   = ("net_pnl",  "count"),
    )
    .sort_values("median_pnl", ascending=False)
)
display(winners_sent)

print("\n── Inconsistent traders: win rate by sentiment ──")
incons_sent = (
    merged2[merged2["winner_segment"] == "Inconsistent"]
    .dropna(subset=["sentiment"])
    .groupby("sentiment")
    .agg(
        win_rate   = ("is_win",   "mean"),
        median_pnl = ("net_pnl",  "median"),
        n_trades   = ("net_pnl",  "count"),
    )
    .sort_values("median_pnl", ascending=False)
)
display(incons_sent)

### **C2. STRATEGY RULE DERIVATION**

In [ ]:
best_pnl_sent  = pnl_by_sent["median_pnl"].idxmax()
worst_pnl_sent = pnl_by_sent["median_pnl"].idxmin()
best_wr_sent   = wr_by_sent["win_rate"].idxmax()
worst_wr_sent  = wr_by_sent["win_rate"].idxmin()

best_pnl_val   = pnl_by_sent.loc[best_pnl_sent,  "median_pnl"]
worst_pnl_val  = pnl_by_sent.loc[worst_pnl_sent, "median_pnl"]
best_wr_val    = wr_by_sent.loc[best_wr_sent,   "win_rate"]
worst_wr_val   = wr_by_sent.loc[worst_wr_sent,  "win_rate"]

# Frequent vs infrequent performance gap across sentiments
freq_gap = (
    merged2.dropna(subset=["sentiment", "freq_segment"])
    .groupby(["freq_segment", "sentiment"])["net_pnl"]
    .median()
    .unstack("sentiment")
)

print("\n── Frequent vs Infrequent median PnL by sentiment ──")
display(freq_gap)

### **C3. STRATEGY VISUALISATION**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sent_order = list(pnl_by_sent.index)  # sorted by median PnL descending

# Chart 1: Median PnL trend across sentiment spectrum
ax = axes[0]
vals  = pnl_by_sent["median_pnl"]
cols  = [COLORS.get(s, "#888780") for s in vals.index]
bars  = ax.bar(vals.index, vals.values, color=cols, edgecolor="white", width=0.55)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
for bar, v in zip(bars, vals.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + abs(vals.values).max() * 0.02,
            f"${v:,.0f}", ha="center", va="bottom", fontsize=9)
ax.set_title("Median daily PnL\nby sentiment", fontweight="bold")
ax.set_ylabel("Median daily PnL (USD)")
ax.tick_params(axis="x", rotation=20)
sns.despine(ax=ax)

# Chart 2: Win rate by sentiment
ax = axes[1]
wr_vals = wr_by_sent["win_rate"] * 100
cols    = [COLORS.get(s, "#888780") for s in wr_vals.index]
bars    = ax.bar(wr_vals.index, wr_vals.values, color=cols, edgecolor="white", width=0.55)
ax.axhline(50, color="black", linewidth=0.8, linestyle="--", label="50% baseline")
for bar, v in zip(bars, wr_vals.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_title("Win rate\nby sentiment", fontweight="bold")
ax.set_ylabel("Win rate (%)")
ax.legend(fontsize=8)
ax.tick_params(axis="x", rotation=20)
sns.despine(ax=ax)

# Chart 3: Winner vs Inconsistent median PnL side-by-side
ax = axes[2]
w_pnl = winners_sent["median_pnl"].reindex(
    [s for s in COLORS if s in winners_sent.index]
)
i_pnl = incons_sent["median_pnl"].reindex(
    [s for s in COLORS if s in incons_sent.index]
)

x      = np.arange(len(w_pnl))
width  = 0.38
ax.bar(x - width / 2, w_pnl.values, width, label="Consistent winners",
       color="#1D9E75", edgecolor="white")
ax.bar(x + width / 2, i_pnl.reindex(w_pnl.index).values, width,
       label="Inconsistent", color="#E24B4A", edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(w_pnl.index, rotation=20, fontsize=9)
ax.set_title("Median PnL: winners\nvs inconsistent traders", fontweight="bold")
ax.set_ylabel("Median trade PnL (USD)")
ax.legend(fontsize=8)
sns.despine(ax=ax)

plt.suptitle("C3 — Evidence for strategy rules", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("C3_strategy_evidence.png", dpi=150, bbox_inches="tight")
plt.show()

### **C4. STRATEGY RULES**

In [ ]:
print("\n" + "=" * 60)
print("STRATEGY RULES")
print("=" * 60)

strategy = pd.DataFrame({
    "Rule": ["Rule 1", "Rule 2"],
    "Name": [
        "Sentiment-adjusted position sizing",
        "Sentiment-aware entry timing for inconsistent traders",
    ],
    "Logic": [
        (
            f"During '{best_pnl_sent}' days (median PnL = ${best_pnl_val:,.0f}), "
            f"traders show higher win rates ({best_wr_val:.1%}) — scale position "
            f"size UP to capture higher-probability setups. "
            f"During '{worst_pnl_sent}' days (median PnL = ${worst_pnl_val:,.0f}), "
            f"reduce position size or sit out to limit downside."
        ),
        (
            "Inconsistent traders underperform most during extreme sentiment days. "
            "Rule: only enter trades on 'Neutral' or moderate sentiment days. "
            "Consistent winners are resilient across all sentiment regimes — "
            "no sentiment filter needed for that segment."
        ),
    ],
    "Applies to": [
        "All traders",
        "Inconsistent traders specifically",
    ],
    "Evidence": [
        "Charts B2, C3 (PnL + win rate by sentiment)",
        "Chart C3 (winner vs inconsistent breakdown)",
    ],
})

display(strategy)

### **C5. RULE BACKTEST**

In [ ]:
print(f"\n── Simple backtest: skip trades on '{worst_pnl_sent}' days ──")

baseline   = merged["net_pnl"].sum()
filtered   = merged[merged["sentiment"] != worst_pnl_sent]["net_pnl"].sum()
skipped_n  = (merged["sentiment"] == worst_pnl_sent).sum()
total_n    = len(merged)

print(f"Total trades     : {total_n:,}")
print(f"Trades skipped   : {skipped_n:,}  ({skipped_n/total_n:.1%} of all trades)")
print(f"Baseline PnL     : ${baseline:,.2f}")
print(f"Filtered PnL     : ${filtered:,.2f}")
print(f"PnL delta        : ${filtered - baseline:+,.2f}")

# Visualise baseline vs filtered cumulative PnL
merged_sorted = merged.sort_values("Timestamp").copy()
merged_sorted["baseline_cum"]  = merged_sorted["net_pnl"].cumsum()
merged_sorted["filtered_pnl"]  = np.where(
    merged_sorted["sentiment"] == worst_pnl_sent, 0, merged_sorted["net_pnl"]
)
merged_sorted["filtered_cum"]  = merged_sorted["filtered_pnl"].cumsum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(merged_sorted["Timestamp"], merged_sorted["baseline_cum"],
        label="Baseline (all trades)", color="#888780", linewidth=1.2, alpha=0.8)
ax.plot(merged_sorted["Timestamp"], merged_sorted["filtered_cum"],
        label=f"Strategy (skip {worst_pnl_sent})", color="#1D9E75", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.set_title(f"C5 — Cumulative PnL: baseline vs sentiment filter\n(skip '{worst_pnl_sent}' days)",
             fontweight="bold")
ax.set_ylabel("Cumulative net PnL (USD)")
ax.set_xlabel("Date")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.savefig("C5_strategy_backtest.png", dpi=150, bbox_inches="tight")
plt.show()

### **C6. RULES OF THUMB SUMMARY**

In [ ]:
print("\n" + "=" * 60)
print("RULES OF THUMB  —  FINAL SUMMARY")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────────┐
│  RULE 1 — Sentiment-adjusted position sizing            │
├─────────────────────────────────────────────────────────┤
│  • On {best_pnl_sent:<15} days → full position size      │
│  • On {worst_pnl_sent:<15} days → reduce size or skip   │
│  Rationale: win rate and median PnL are materially      │
│  different across sentiment regimes (see B2, C3).       │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│  RULE 2 — Entry timing for inconsistent traders         │
├─────────────────────────────────────────────────────────┤
│  • Trade only on Neutral / moderate sentiment days      │
│  • Avoid Extreme Fear & Extreme Greed days              │
│  Rationale: inconsistent traders show the largest PnL   │
│  swings on extreme sentiment days. Consistent winners   │
│  are not affected — no filter needed for them.          │
└─────────────────────────────────────────────────────────┘
""")

## **BONUS**
[Predictive Model](#scrollTo=BmsFxeUTyrpz)

[Clustering Model](#scrollTo=mx-Yxh_D2CRb)

[Interactive Charts](#scrollTo=Gw37XnVG3g_l)

In [ ]:
!pip install plotly ipywidgets scikit-learn --quiet

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display as ipy_display
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

print("Bonus libraries ready.")

### **B1 — PREDICTIVE MODEL**

**B1 a. Build Features**

In [ ]:
# Goal: predict whether a trader's NEXT DAY will be profitable using today's behavior + sentiment as features.

print("\n" + "=" * 60)
print("BONUS 1 — PREDICTIVE MODEL")
print("=" * 60)

# Target: was the NEXT trading day profitable for this account?

daily_feat = daily.dropna(subset=["sentiment"]).copy()

sent_order = {
    "Extreme Fear": 0, "Fear": 1, "Neutral": 2,
    "Greed": 3, "Extreme Greed": 4
}
daily_feat["sentiment_score"] = daily_feat["sentiment"].map(sent_order)


daily_feat = daily_feat.sort_values(["account", "date_only"])


daily_feat["next_pnl"]       = daily_feat.groupby("account")["daily_pnl"].shift(-1)
daily_feat["target_profitable"] = (daily_feat["next_pnl"] > 0).astype(int)


daily_feat = daily_feat.dropna(subset=["next_pnl"])

feature_candidates = [
    "daily_pnl", "trade_count", "long_ratio",
    "win_rate", "avg_size_usd", "sentiment_score",
]
features = [c for c in feature_candidates if c in daily_feat.columns]

model_df = daily_feat[features + ["target_profitable"]].dropna()

print(f"Model dataset: {len(model_df):,} rows  |  features: {features}")
print(f"Class balance — profitable: {model_df['target_profitable'].mean():.1%}")

**B1 b. TRAIN/TEST SPLIT**

In [ ]:
X = model_df[features]
y = model_df["target_profitable"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

**B1 C. TRAIN THREE MODELS, PICK BEST**

In [ ]:
models = {
    "Logistic Regression":    LogisticRegression(max_iter=500, random_state=42),
    "Random Forest":          RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting":      GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    X_use = X_train_s if name == "Logistic Regression" else X_train
    scores = cross_val_score(model, X_use, y_train, cv=cv, scoring="roc_auc")
    results[name] = {"mean_auc": scores.mean(), "std_auc": scores.std(), "model": model}
    print(f"{name:<25}  CV AUC = {scores.mean():.4f} ± {scores.std():.4f}")

best_name  = max(results, key=lambda k: results[k]["mean_auc"])
best_model = results[best_name]["model"]
print(f"\nBest model: {best_name}")

**B1 d. FINAL EVALUATION**

In [ ]:
X_fit = X_train_s if best_name == "Logistic Regression" else X_train
X_eval = X_test_s if best_name == "Logistic Regression" else X_test
best_model.fit(X_fit, y_train)
y_pred      = best_model.predict(X_eval)
y_pred_prob = best_model.predict_proba(X_eval)[:, 1]

print(f"\n── Classification report ({best_name}) ──")
print(classification_report(y_test, y_pred, target_names=["Loss day", "Profit day"]))

auc = roc_auc_score(y_test, y_pred_prob)
print(f"Test AUC: {auc:.4f}")

**B1 e. PLOTS: confusion matrix + ROC + feature importance**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Loss", "Profit"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title(f"Confusion matrix\n{best_name}", fontweight="bold")

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color="#1D9E75", linewidth=2, label=f"AUC = {auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=0.8)
axes[1].set_title("ROC curve", fontweight="bold")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].legend()
sns.despine(ax=axes[1])

# Feature importance
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=features).sort_values()
    imp.plot.barh(ax=axes[2], color="#378ADD", edgecolor="white")
    axes[2].set_title("Feature importance", fontweight="bold")
    axes[2].set_xlabel("Importance")
    sns.despine(ax=axes[2])
elif hasattr(best_model, "coef_"):
    imp = pd.Series(np.abs(best_model.coef_[0]), index=features).sort_values()
    imp.plot.barh(ax=axes[2], color="#378ADD", edgecolor="white")
    axes[2].set_title("Feature coefficients (abs)", fontweight="bold")
    axes[2].set_xlabel("|Coefficient|")
    sns.despine(ax=axes[2])

plt.suptitle(f"Bonus 1 — Predictive model: {best_name}", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("Bonus1_predictive_model.png", dpi=150, bbox_inches="tight")
plt.show()

print("""
── Model interpretation ──
AUC > 0.55 : sentiment + behavior features carry predictive signal.
AUC ≈ 0.50 : next-day profitability is largely random (efficient market).
Feature importance shows WHICH inputs drive the prediction most.
""")

### **BONUS 2 — CLUSTERING (BEHAVIORAL ARCHETYPES)**

In [ ]:
# Goal: let the data group traders naturally — no manual cut-offs.

print("\n" + "=" * 60)
print("BONUS 2 — CLUSTERING")
print("=" * 60)

# ── B2a. CLUSTER FEATURES ─────────────────────────────────────────────────────

cluster_candidates = [
    "win_rate", "avg_trade_size", "long_ratio",
    "pnl_volatility", "total_trades", "pnl_per_trade",
]
cluster_cols = [c for c in cluster_candidates if c in acct.columns]

cluster_df = acct[["account"] + cluster_cols].dropna().copy()
print(f"Clustering on {len(cluster_df):,} accounts  |  features: {cluster_cols}")

# Scale
sc2      = StandardScaler()
X_clust  = sc2.fit_transform(cluster_df[cluster_cols])

**B2 b. CHOOSE K WITH ELBOW METHOD**

In [ ]:
inertias = []
K_range  = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), inertias, marker="o", color="#378ADD", linewidth=2)
ax.set_title("Elbow method — choose number of clusters", fontweight="bold")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia")
sns.despine()
plt.tight_layout()
plt.savefig("Bonus2_elbow.png", dpi=150, bbox_inches="tight")
plt.show()

**B2 c. FIT FINAL MODEL**

In [ ]:
K_FINAL = 5
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
cluster_df["cluster"] = km_final.fit_predict(X_clust)

**B2 d. PROFILE EACH CLUSTER**

In [ ]:
profile = (
    cluster_df.groupby("cluster")[cluster_cols]
    .mean()
    .round(4)
)
profile["n_accounts"] = cluster_df.groupby("cluster").size()

print("\n── Cluster profiles ──")
display(profile)

# Auto-label clusters based on win_rate and pnl_per_trade
def label_cluster(row):
    wr  = row.get("win_rate", 0.5)
    ppt = row.get("pnl_per_trade", 0)
    vol = row.get("pnl_volatility", 0)
    freq = row.get("total_trades", 0)

    if wr > 0.55 and ppt > 0:
        return "Consistent winners"
    elif freq > profile["total_trades"].median() and vol > profile["pnl_volatility"].median():
        return "High-frequency risk-takers"
    elif ppt < 0 and wr < 0.45:
        return "Chronic losers"
    else:
        return "Cautious / infrequent"

profile["archetype"] = profile.apply(label_cluster, axis=1)
print("\n── Archetypes ──")
display(profile[["archetype", "n_accounts"]])

# Map labels back to cluster_df
archetype_map = profile["archetype"].to_dict()
cluster_df["archetype"] = cluster_df["cluster"].map(archetype_map)

# Merge archetypes into merged for downstream use
merged3 = merged.merge(
    cluster_df[["account", "cluster", "archetype"]],
    on="account", how="left"
)

**B2 e. VISUALISE CLUSTERS**

In [ ]:
pca      = PCA(n_components=2, random_state=42)
coords   = pca.fit_transform(X_clust)
cluster_df["pca1"] = coords[:, 0]
cluster_df["pca2"] = coords[:, 1]

var1, var2 = pca.explained_variance_ratio_ * 100

palette = ["#378ADD", "#1D9E75", "#E24B4A", "#EF9F27",
           "#534AB7", "#D85A30", "#639922", "#D4537E"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
ax = axes[0]
for c, grp in cluster_df.groupby("cluster"):
    ax.scatter(grp["pca1"], grp["pca2"],
               label=archetype_map.get(c, f"Cluster {c}"),
               color=palette[c % len(palette)], alpha=0.6, s=20)
ax.set_title("Trader clusters (PCA projection)", fontweight="bold")
ax.set_xlabel(f"PC1 ({var1:.1f}% variance)")
ax.set_ylabel(f"PC2 ({var2:.1f}% variance)")
ax.legend(fontsize=8, markerscale=2)
sns.despine(ax=ax)


ax = axes[1]
profile_plot = profile[cluster_cols].copy()

for col in cluster_cols:
    rng = profile_plot[col].max() - profile_plot[col].min()
    profile_plot[col] = (profile_plot[col] - profile_plot[col].min()) / (rng if rng else 1)

profile_plot.T.plot(kind="bar", ax=ax,
                    color=palette[:K_FINAL], edgecolor="white", width=0.7)
ax.set_title("Normalised cluster profiles", fontweight="bold")
ax.set_ylabel("Normalised value (0–1)")
ax.set_xticklabels(cluster_cols, rotation=30, ha="right", fontsize=9)
ax.legend([archetype_map.get(i, f"C{i}") for i in range(K_FINAL)],
          fontsize=8, loc="upper right")
sns.despine(ax=ax)

plt.suptitle("Bonus 2 — Behavioral archetypes", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("Bonus2_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

**B2 f. ARCHETYPE x SENTIMENT**

In [ ]:
arch_sent = (
    merged3.dropna(subset=["sentiment", "archetype"])
    .groupby(["archetype", "sentiment"])
    .agg(median_pnl=("net_pnl", "median"), win_rate=("is_win", "mean"))
    .reset_index()
)

pivot_arch = arch_sent.pivot(index="archetype", columns="sentiment", values="median_pnl")

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot_arch, annot=True, fmt=".0f", cmap="RdYlGn",
            linewidths=0.5, ax=ax)
ax.set_title("Bonus 2 — Median PnL: archetype × sentiment", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("Bonus2_archetype_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## **BONUS 3 — INTERACTIVE CHARTS**

**CHART 1: Interactive PnL distribution by sentiment**

In [ ]:
print("\n" + "=" * 60)
print("BONUS 3 — INTERACTIVE CHARTS")
print("=" * 60)

fig1 = px.box(
    daily.dropna(subset=["sentiment"]),
    x="sentiment", y="daily_pnl",
    color="sentiment",
    color_discrete_map=COLORS,
    points="outliers",
    title="Daily PnL distribution by sentiment (interactive)",
    labels={"daily_pnl": "Daily PnL (USD)", "sentiment": "Sentiment"},
    category_orders={"sentiment": list(sent_order.keys())},
)
fig1.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
fig1.update_layout(showlegend=False, height=450)
fig1.show()

**CHART 2: Scatter — win rate vs total PnL, coloured by archetype**

In [ ]:
if "archetype" in cluster_df.columns:
    scatter_df = acct.merge(
        cluster_df[["account", "archetype"]], on="account", how="left"
    ).dropna(subset=["archetype"])

    fig2 = px.scatter(
        scatter_df,
        x="win_rate", y="total_pnl",
        color="archetype",
        size="total_trades",
        hover_data=["account", "avg_trade_size", "pnl_volatility"],
        title="Trader archetypes — win rate vs total PnL (bubble = trade count)",
        labels={"win_rate": "Win rate", "total_pnl": "Total PnL (USD)"},
        opacity=0.7,
    )
    fig2.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
    fig2.add_vline(x=0.5, line_dash="dash", line_color="black", line_width=1)
    fig2.update_layout(height=500)
    fig2.show()

**CHART 3: ipywidgets — filter by sentiment, see cumulative PnL**

In [ ]:
merged_sorted2 = merged.sort_values("Timestamp").copy()
merged_sorted2["cum_pnl"] = merged_sorted2["net_pnl"].cumsum()
all_sentiments = sorted(merged_sorted2["sentiment"].dropna().unique().tolist())

sent_widget = widgets.SelectMultiple(
    options=all_sentiments,
    value=all_sentiments,
    description="Sentiment:",
    layout=widgets.Layout(height="130px", width="220px"),
)
out = widgets.Output(layout={'border': '1px solid #ccc', 'width': '100%', 'min_height': '450px'})

def update_chart(change):
    selected = list(sent_widget.value)
    with out:
        out.clear_output(wait=True)
        filtered = merged_sorted2[merged_sorted2["sentiment"].isin(selected)].copy()
        filtered["cum_pnl_filtered"] = filtered["net_pnl"].cumsum()

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=merged_sorted2["Timestamp"],
            y=merged_sorted2["cum_pnl"],
            name="All trades (baseline)",
            line=dict(color="#888780", width=1.5, dash="dot"),
        ))
        fig.add_trace(go.Scatter(
            x=filtered["Timestamp"],
            y=filtered["cum_pnl_filtered"],
            name=f"Selected: {', '.join(selected)}",
            line=dict(color="#1D9E75", width=2),
        ))
        fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=0.8)
        fig.update_layout(
            title="Cumulative PnL — filter by sentiment",
            xaxis_title="Date",
            yaxis_title="Cumulative net PnL (USD)",
            height=420,
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        )
        fig.show()

sent_widget.observe(update_chart, names="value")

print("Select sentiments below to filter the cumulative PnL chart:")
ipy_display(widgets.HBox([sent_widget, out]))
update_chart(None)  # render on load

**CHART 4: ipywidgets — per-account PnL explorer**

In [ ]:
top_accounts = (
    acct.nlargest(30, "total_trades")["account"].tolist()
)

acct_widget = widgets.Dropdown(
    options=top_accounts,
    description="Account:",
    layout=widgets.Layout(width="380px"),
)
out2 = widgets.Output()

def update_account(change):
    with out2:
        out2.clear_output(wait=True)
        acc = acct_widget.value
        acc_trades = (
            merged[merged["account"] == acc]
            .sort_values("Timestamp")
            .copy()
        )
        acc_trades["cum_pnl"] = acc_trades["net_pnl"].cumsum()

        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            subplot_titles=("Cumulative PnL", "Trade PnL per trade"),
            row_heights=[0.6, 0.4],
        )
        fig.add_trace(go.Scatter(
            x=acc_trades["Timestamp"], y=acc_trades["cum_pnl"],
            name="Cum PnL", line=dict(color="#378ADD", width=2),
        ), row=1, col=1)

        colors = ["#1D9E75" if v > 0 else "#E24B4A"
                  for v in acc_trades["net_pnl"]]
        fig.add_trace(go.Bar(
            x=acc_trades["Timestamp"], y=acc_trades["net_pnl"],
            name="Trade PnL", marker_color=colors,
        ), row=2, col=1)

        fig.add_hline(y=0, line_dash="dash", line_color="black",
                      line_width=0.8, row=1, col=1)
        fig.update_layout(
            title=f"Account explorer — {acc[:12]}...",
            height=520, showlegend=False,
        )
        fig.show()

acct_widget.observe(update_account, names="value")

print("\nSelect an account to explore its trade history:")
ipy_display(widgets.VBox([acct_widget, out2]))
update_account(None)  # render on load